# Viscosity profile, nondimensionalized with GeoParams

This version keeps all internal length, temperature, pressure, strain-rate, rheology, and viscosity values nondimensional. Values are dimensionalized only when plotting or printing diagnostics.

In [ ]:
using CairoMakie, GeoParams

n = 256

# Characteristic scales used by the profile and rheologies.
CharDim = GEO_units(;
    length = 2890km,
    viscosity = 1.0e21Pa * s,
    temperature = 1000K,
)

lz = nondimensionalize(2890km, CharDim)
z = LinRange(-lz, 0.0, n)

dimstrip(x, unit) = ustrip(dimensionalize(x, unit, CharDim))
z_km = [dimstrip(zi, km) for zi in z]
z_m = [dimstrip(zi, m) for zi in z];

: 

## Initial thermal and pressure profiles

In [ ]:
# Smooth switch
S(ξ) = 0.5 * (1.0 + tanh(ξ))

function T_field(
    depth;
    Lz = nondimensionalize(2890km, CharDim),
    Ttop = nondimensionalize(273K, CharDim),
    Tbot = nondimensionalize(2800K, CharDim),
    Tm = nondimensionalize((1350 + 273)K, CharDim),
    delta_t = nondimensionalize(110km, CharDim),
    delta_b = nondimensionalize(100km, CharDim),
    w_t = nondimensionalize(20km, CharDim),
    w_b = nondimensionalize(20km, CharDim),
)
    s_t = S((depth - delta_t) / w_t)
    s_b = S(((Lz - delta_b) - depth) / w_b)

    T_topBL = Ttop + (Tm - Ttop) * (depth / delta_t)
    T_botBL = Tm + (Tbot - Tm) * ((depth - (Lz - delta_b)) / delta_b)

    return (1.0 - s_t) * T_topBL + (s_t * s_b) * Tm + (1.0 - s_b) * T_botBL
end

T = T_field.(-z)

ρ0 = 3300kg / m^3
g = 9.81m / s^2
P = [nondimensionalize(ρ0 * g * max(-zi_m, 0.0) * m, CharDim) for zi_m in z_m]

T_K = [dimstrip(Ti, K) for Ti in T]
P_GPa = [dimstrip(Pi, GPa) for Pi in P]

fig = Figure(size = (900, 700), fontsize = 22)
ax1 = Axis(fig[1, 1], aspect = 2 / 3, ylabel = "z (km)", xlabel = "T (C)")
ax2 = Axis(fig[1, 2], aspect = 2 / 3, ylabel = "z (km)", xlabel = "P (GPa)")
lines!(ax1, T_K .- 273.15, z_km)
lines!(ax2, P_GPa, z_km)
fig

## Phases as a function of depth

In [ ]:
function get_phase(z_nd)
    d = -z_nd
    if d < nondimensionalize(20km, CharDim)
        return 1 # upper crust
    elseif d < nondimensionalize(40km, CharDim)
        return 2 # lower crust
    elseif d < nondimensionalize(660km, CharDim)
        return 3 # upper mantle
    else
        return 4 # lower mantle
    end
end

phase = get_phase.(z)

fig = Figure(size = (500, 700), fontsize = 22)
ax = Axis(fig[1, 1], aspect = 2 / 3, ylabel = "z (km)", xlabel = "phase")
lines!(ax, phase, z_km)
fig

## Rheology

Rheologies from Table S2 of [Li et al. 2026](https://agupubs.onlinelibrary.wiley.com/doi/epdf/10.1029/2025JB032510?saml_referrer). The creep laws are declared with physical units and then nondimensionalized with `CharDim`.

### Crust

In [ ]:
rheo_upper_crust_dim = DislocationCreep(
    A = 5.0e-18Pa^(-23 // 10) / s,
    n = 2.3NoUnits,
    E = 154.0e3J / mol,
    V = 8.0e-6m^3 / mol,
    r = 0.0NoUnits,
    R = 8.3145J / mol / K,
)

rheo_lower_crust_dim = DislocationCreep(
    A = 2.0e-23Pa^(-32 // 10) / s,
    n = 3.2NoUnits,
    E = 238.0e3J / mol,
    V = 8.0e-6m^3 / mol,
    r = 0.0NoUnits,
    R = 8.3145J / mol / K,
);

### Mantle

In [ ]:
disl_mantle_dim = DislocationCreep(
    A = 1.1e-17Pa^(-35 // 10) / s,
    n = 3.5NoUnits,
    E = 530.0e3J / mol,
    V = 8.0e-6m^3 / mol,
    r = 0.0NoUnits,
    R = 8.3145J / mol / K,
    Apparatus = Invariant,
)

diff_mantle_dim = DiffusionCreep(
    A = 2.2e-10Pa^(-1) * s^(-1),
    E = 375.0e3J / mol,
    V = 4.0e-6m^3 / mol,
    n = 1.0NoUnits,
    r = 0.0NoUnits,
    p = 0NoUnits,
)

rheo_mantle_dim = CompositeRheology(disl_mantle_dim, diff_mantle_dim)

rheo_lower_mantle_dim = DiffusionCreep(
    A = 2.2e-13Pa^(-1) * s^(-1),
    E = 300.0e3J / mol,
    V = 2.0e-6m^3 / mol,
    n = 1.0NoUnits,
    r = 0.0NoUnits,
    p = 0NoUnits,
    Apparatus = Invariant,
);

### Nondimensional rheology

In [ ]:
rheology_dim = (
    rheo_upper_crust_dim,
    rheo_lower_crust_dim,
    rheo_mantle_dim,
    rheo_lower_mantle_dim,
)

rheology = map(r -> nondimensionalize(r, CharDim), rheology_dim);

## Compute viscosity profile

The strain rate, pressure, temperature, rheology, and viscosity limits are nondimensional in this loop.

In [ ]:
εII = nondimensionalize(1.0e-15 / s, CharDim)
η_limits = nondimensionalize((1.0e18Pa * s, 1.0e23Pa * s), CharDim)

η = zeros(n)
for i in eachindex(z)
    phaseᵢ = phase[i]
    args = (; T = T[i], P = P[i], dt = Inf)
    η[i] = clamp(
        compute_viscosity_εII(rheology[phaseᵢ], εII, args) * 2,
        η_limits...,
    )
end

η_Pa_s = [dimstrip(ηi, Pa * s) for ηi in η];

In [ ]:
fig = Figure(size = (1200, 900), fontsize = 24)
ax1 = Axis(fig[1, 1], aspect = 2 / 3, ylabel = "z (km)", xlabel = "T (C)")
ax2 = Axis(fig[1, 2], aspect = 2 / 3, ylabel = "z (km)", xlabel = "log10(η / Pa s)")

lines!(ax1, T_K .- 273.15, z_km, linewidth = 2)
lines!(ax2, log10.(η_Pa_s), z_km, color = :black, linewidth = 2, label = "GeoParams ND")

xlims!(ax2, 18.5, 23.5)
axislegend(ax2, position = :lb)
fig

## Point checks

In [ ]:
Ti = nondimensionalize(1600K, CharDim)
Pi = nondimensionalize(3300kg / m^3 * 9.81m / s^2 * 300km, CharDim)
args = (; T = Ti, P = Pi, dt = Inf)

η_mantle = compute_viscosity_εII(rheology[3], εII, args)
η_disl = compute_viscosity_εII(rheology[3].elements[1], εII, args)
η_diff = compute_viscosity_εII(rheology[3].elements[2], εII, args)

(;
    T_nd = Ti,
    P_nd = Pi,
    εII_nd = εII,
    η_mantle_nd = η_mantle,
    η_mantle_Pa_s = dimstrip(η_mantle, Pa * s),
    η_disl_Pa_s = dimstrip(η_disl, Pa * s),
    η_diff_Pa_s = dimstrip(η_diff, Pa * s),
)